# Chapitre 18 - Exercices 18.16 à 18.20 : tests multiples

Ces cinq exercices portent sur la section 18.7 du livre (évaluation de l'importance de
chaque variable et problème des **tests multiples**) :

- **18.16** : la méthode de Bonferroni contrôle le taux d'erreur par famille (FWER) ;
- **18.17** : équivalence entre la procédure de Benjamini–Hochberg et l'estimation
  « plug-in » du taux de fausses découvertes (FDR) ;
- **18.18** : expression bayésienne du pFDR à l'aide de l'erreur de type I et de la
  puissance ;
- **18.19** : étude complète (FDR plug-in, BH, estimation de $\pi_0$) sur des données de
  micro-puces ;
- **18.20** : démonstration du résultat (18.53), $\text{pFDR}(\Gamma) = \Pr(Z = 0 \mid t \in \Gamma)$.

Les fonctions utilisées sont dans `esl/ch18_grande_dimension_complements.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch18_grande_dimension_complements as g18
set_style()

### Rappels communs : vocabulaire des tests multiples

On teste $M$ hypothèses nulles $H_{01}, \dots, H_{0M}$ (par exemple « le gène $j$ a la
même expression moyenne dans les deux groupes »). Chaque test fournit une **p-valeur**
$p_j$. Sous $H_{0j}$, $p_j$ suit (au moins approximativement) la loi uniforme sur
$[0, 1]$, de sorte que $\Pr(p_j < u) = u$ : rejeter quand $p_j < u$ donne une erreur de
type I égale à $u$.

Le tableau 18.5 du livre résume les issues possibles :

| | non rejetée | rejetée | total |
|---|---|---|---|
| $H_0$ vraie | $U$ | $V$ | $M_0$ |
| $H_0$ fausse | $T$ | $S$ | $M_1$ |
| total | $M - R$ | $R$ | $M$ |

$V$ est le nombre de **faux positifs** et $R$ le nombre total de rejets.

- **FWER** (taux d'erreur par famille) : $\Pr(V \ge 1)$.
- **FDR** (taux de fausses découvertes, éq. (18.43)) : $E(V/R)$, avec la convention
  $V/R = 0$ si $R = 0$.
- **pFDR** (FDR positif, éq. (18.50)) : $E(V/R \mid R > 0)$.

## Exercice 18.16 : Méthode de Bonferroni

**Énoncé (reformulé).** On teste $M$ hypothèses nulles $H_{0j}$ de p-valeurs $p_j$. Soit
$A$ l'événement « au moins une hypothèse nulle est rejetée à tort » et $A_j$ l'événement
« la $j$-ème hypothèse nulle est rejetée à tort ». La méthode de Bonferroni rejette
$H_{0j}$ si $p_j < \alpha/M$.

(a) Montrer que $\Pr(A) \le \alpha$. *Indication :*
$\Pr(A_j \cup A_{j'}) = \Pr(A_j) + \Pr(A_{j'}) - \Pr(A_j \cap A_{j'})$.

(b) Si les hypothèses sont indépendantes,
$\Pr(A) = 1 - \Pr(A^c) = 1 - \prod_j \Pr(A_j^c) = 1 - (1 - \alpha/M)^M$. En déduire que
$\Pr(A) \approx \alpha$ dans ce cas.

### (a) Inégalité de la réunion

**Étape 1 : deux événements.** D'après l'indication, et puisqu'une probabilité est
positive,
$\Pr(A_j \cup A_{j'}) = \Pr(A_j) + \Pr(A_{j'}) - \Pr(A_j \cap A_{j'}) \le \Pr(A_j) + \Pr(A_{j'})$.

**Étape 2 : récurrence.** Supposons $\Pr\big(\bigcup_{j=1}^{m} A_j\big) \le \sum_{j=1}^m \Pr(A_j)$.
En appliquant l'étape 1 aux deux événements $B = \bigcup_{j=1}^{m} A_j$ et $A_{m+1}$ :

$$
\Pr\Big(\bigcup_{j=1}^{m+1} A_j\Big) \le \Pr(B) + \Pr(A_{m+1}) \le \sum_{j=1}^{m+1} \Pr(A_j).
$$

C'est l'**inégalité de Boole** : $\Pr\big(\bigcup_j A_j\big) \le \sum_j \Pr(A_j)$.

**Étape 3 : probabilité de chaque fausse alarme.** Si $H_{0j}$ est fausse, l'événement
$A_j$ est impossible (on ne peut pas la rejeter *à tort*) : $\Pr(A_j) = 0$. Si $H_{0j}$ est
vraie, $p_j$ est uniforme et $\Pr(A_j) = \Pr(p_j < \alpha/M) = \alpha/M$.

**Conclusion.** Avec $M_0 \le M$ hypothèses nulles vraies,

$$
\Pr(A) = \Pr\Big(\bigcup_{j=1}^M A_j\Big) \le \sum_{j=1}^M \Pr(A_j) = M_0\,\frac{\alpha}{M} \le \alpha .
$$

Aucune hypothèse d'indépendance n'a été utilisée : la garantie est valable quelle que
soit la dépendance entre les tests.